In [ ]:
from pathlib import Path
import pickle
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = Path("..") / "data" / "raw" / "WESAD" / "WESAD"
print(DATA_DIR.resolve())
print(DATA_DIR.exists())

subjects = sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir())
print(subjects)

In [ ]:
with open(DATA_DIR / "S3" / "S3.pkl", "rb") as f:
    data = pickle.load(f, encoding="latin1")

print(type(data))
print(data.keys())
print(data["signal"].keys())
print(data["signal"]["wrist"].keys())
print(data["signal"]["wrist"]["ACC"].shape)

In [ ]:
DATA_DIR

In [ ]:
sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])

In [ ]:
FS = {"ACC": 32, "BVP": 64, "EDA": 4, "TEMP": 4, "label": 700}

for name, signal in data["signal"]["wrist"].items():
    n = signal.shape[0]
    print(f"{name:5s} shape={str(signal.shape):14s} duration={n / FS[name] / 60:6.1f} min")

labels = data["label"]
print(f"label shape={labels.shape} duration={len(labels) / FS['label'] / 60:6.1f} min")

In [ ]:
LABEL_NAMES = {0: "not defined / transient", 1: "baseline", 2: "stress",
               3: "amusement", 4: "meditation", 5: "ignore", 6: "ignore", 7: "ignore"}

values, counts = np.unique(labels, return_counts=True)
for v, c in zip(values, counts):
    print(f"{v}: {LABEL_NAMES[v]:25s} {c / FS['label'] / 60:6.1f} min")

In [ ]:
eda = data["signal"]["wrist"]["EDA"][:, 0]
t_eda = np.arange(len(eda)) / FS["EDA"] / 60
t_label = np.arange(len(labels)) / FS["label"] / 60

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(t_eda, eda, color="tab:blue")
ax1.set_xlabel("time (min)")
ax1.set_ylabel("EDA (µS)", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(t_label[::700], labels[::700], color="tab:red", alpha=0.6)
ax2.set_ylabel("label", color="tab:red")
plt.title("S2: wrist EDA with study condition")
plt.show()

In [ ]:
eda = data["signal"]["wrist"]["EDA"][:, 0]
t_eda = np.arange(len(eda)) / FS["EDA"] / 60
t_label = np.arange(len(labels)) / FS["label"] / 60

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.plot(t_eda, eda, color="tab:blue")
ax1.set_xlabel("time (min)")
ax1.set_ylabel("EDA (µS)", color="tab:blue")

ax2 = ax1.twinx()
ax2.plot(t_label[::700], labels[::700], color="tab:red", alpha=0.6)
ax2.set_ylabel("label", color="tab:red")
plt.title("S3: wrist EDA with study condition")
plt.show()

In [ ]:
# labels
quest_path = DATA_DIR / "S14" / "S14_quest.csv"
print(quest_path.read_text())

In [ ]:
import pandas as pd
from wearable_affect.data import DEFAULT_DATA_DIR, list_subjects, load_self_reports

subjects = list_subjects(DEFAULT_DATA_DIR)
reports = pd.concat([load_self_reports(DEFAULT_DATA_DIR, s) for s in subjects])
wide = reports.pivot(index="subject_id", columns="condition", values=["valence", "arousal"])

check = pd.DataFrame({
    "arousal_base": wide["arousal"]["Base"],
    "arousal_tsst": wide["arousal"]["TSST"],
    "valence_base": wide["valence"]["Base"],
    "valence_tsst": wide["valence"]["TSST"],
}).reindex(subjects)
check["arousal_change"] = check["arousal_tsst"] - check["arousal_base"]
check["valence_change"] = check["valence_tsst"] - check["valence_base"]
check

In [ ]:
from scipy.stats import spearmanr

baseline = per_subject("all_handcrafted-lightgbm")
scl = features.groupby(["subject_id", "target"])["scl_mean"].mean().unstack()

check["auroc"] = baseline["auroc"]
check["scl_change"] = scl[1] - scl[0]

print(spearmanr(check["arousal_change"], check["auroc"]))
print(spearmanr(check["arousal_change"], check["scl_change"]))